In [ ]:
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from selenium import webdriver
from time import sleep
import os
import requests
import warnings
warnings.filterwarnings("ignore")



In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'ZA JSE' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
#writer = ExcelWriter(filename)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running ZA JSE Web Scraping Tool v.1.1


In [3]:

# Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
         "download.prompt_for_download": False,
         "download.default_directory": tempfolder}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [ ]:

# Creating dictionary with Regcodes and their respective URLs
regdict = {
    'ZA JSE 1': 'Equity Issuer',
    'ZA JSE 2': 'ETF Issuer',
    'ZA JSE 3': 'Warrant Issuer',
    'ZA JSE 4': 'Interest Rate Issuer',
    'ZA JSE 5': 'Debt Issuer',
    'ZA JSE 6': 'Structured Product Issuer',
    'ZA JSE 7': 'ETN Issuer',
    'ZA JSE 8': 'Asset Backed Securities (ABS) Issuer',
    'ZA JSE 9': 'Hybrid Issuer',
    'ZA JSE 10': 'Actively Managed Certificate Issuer',
    'ZA JSE 11': 'Actively Managed ETF Issuer'
    }

Typology={

    'ZA JSE 1': 'Equity Issuer',
    'ZA JSE 2': 'ETF Issuer',
    'ZA JSE 3': 'Warrant Issuer',
    'ZA JSE 4': 'Interest Rate Issuer',
    'ZA JSE 5': 'Debt Issuer',
    'ZA JSE 6': 'Structured Product Issuer',
    'ZA JSE 7': 'ETN Issuer',
    'ZA JSE 8': 'Asset Backed Securities (ABS) Issuer',
    'ZA JSE 9': 'Hybrid Issuer',
    'ZA JSE 10': 'Actively Managed Certificate Issuer',
    'ZA JSE 11': 'Actively Managed ETF Issuer'

        }

# Creating dictionary to containg regulators data and then be converted to a pandas' DataFrame
sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [],
           'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
           'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [],
           'Address_1': [], 'Address_2': [], 'City': [],
           'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [],
           'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
           'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [],
           'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
           'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [],
           'Zip - Mother company': [], 'Cntry - Mother company': [],
           'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')
# main_url = 'https://clientportal.jse.co.za/companies-and-financial-instruments'


In [5]:

url = "https://clientportal.jse.co.za/_vti_bin/JSE/CustomerRoleService.svc/GetAllIssuers"


headers = {
    "Content-Type": "application/json; charset=UTF-8",
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://clientportal.jse.co.za",
    "Referer": "https://clientportal.jse.co.za/companies-and-financial-instruments",
    "User-Agent": "Mozilla/5.0"
}

session = requests.Session()

#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict



In [ ]:
for reg, (listname) in regdict.items():
    print('Working with {}'.format(reg))
    sleep(4)


    payload = {
    "filterLongName": "",
    "filterType": listname   # or listname
    }

    response = session.post(url, json=payload, headers=headers, timeout=30,verify=False)

    # print(response.status_code)
    # print(response.json())
    data_json = response.json()

    for data in data_json  :
        if data['Status'] == 'Suspended':
            print(f"Suspended company found: {data['LongName']}")
            continue
        name = data['LongName']
        address_1 = data['PhysicalAddress'] if data['PhysicalAddress'] is not None else ''
        address_2 = data['PostalAddress'] if data['PostalAddress'] is not None else ''
        registration_number = data['RegistrationNumber'] if data['RegistrationNumber'] is not None else ''
        role_description = data['RoleDescription'] if data['RoleDescription'] is not None else ''
        tele_phone = data['TelephoneNumber'] if data['TelephoneNumber'] is not None else ''
        fax_number = data['FaxNumber'] if data['FaxNumber'] is not None else ''
        email = data['EmailAddress'] if data['EmailAddress'] is not None else ''
        website = data['Website'] if data['Website'] is not None else ''
        master_id = data['MasterID'] if data['MasterID'] is not None else ''
    
        sqldict['Name'].append(name)
        sqldict['InternalID_1'].append(registration_number)
        if registration_number:
            sqldict['InternalID_1_type'].append('Registration Number')
        sqldict['InternalID_2'].append(master_id)
        if master_id:
            sqldict['InternalID_2_type'].append('Master ID')
        sqldict['ListProcessDate'].append(processdate)
        sqldict['Phone'].append(tele_phone)
        sqldict['Email'].append(email)
        sqldict['Fax'].append(fax_number)
        sqldict['Website'].append(website)
        sqldict['Address_1'].append(address_1)
        sqldict['Address_2'].append(address_2)
        sqldict['RegCtry'].append(reg.split()[0])
        sqldict['RegCode'].append(reg.split()[1])
        sqldict['ListCode'].append(reg.split()[2])
        sqldict['RegulationType'].append('Regulated')
        sqldict['ListName'].append(Typology[reg])
        sqldict['Typology'].append(role_description)
        sqldict = bourange_same_length_array(sqldict)
   
    

Working with ZA JSE 1
Suspended company found: AFRICAN DAWN CAPITAL LIMITED
Suspended company found: aReit PROP LIMITED
Suspended company found: EFORA ENERGY LIMITED
Suspended company found: ELLIES HOLDINGS LIMITED
Suspended company found: IMPACT PROPERTY FUND LIMITED
Suspended company found: KIBO ENERGY PLC
Suspended company found: SABLE EXPLORATION AND MINING LIMITED
Suspended company found: SAIL MINING GROUP LIMITED
Suspended company found: SALUNGANO GROUP LIMITED
Suspended company found: SEBATA HOLDINGS LIMITED
Suspended company found: TONGAAT HULETT LIMITED
Suspended company found: TRUSTCO GROUP HOLDINGS LIMITED
Suspended company found: TRUSTCO GROUP HOLDINGS LIMITED
Suspended company found: WESIZWE PLATINUM LIMITED
Working with ZA JSE 2
Working with ZA JSE 3
Working with ZA JSE 4
Suspended company found: MARTIUS (RF) LIMITED
Working with ZA JSE 5
Suspended company found: FIRSTRAND BANK LIMITED
Suspended company found: INVESTEC LIMITED
Suspended company found: SOAPSTONE INVESTMENT

In [7]:
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)

# Remove duplicates based on Name, InternalID_1, InternalID_1_type, and ListCode
# df = df.drop_duplicates(subset=['Name', 'InternalID_1', 'InternalID_1_type', 'ListCode'], keep='first')

df.to_excel(filename, index=False)
driver.quit()
    
    